# M3P2 - Pseudo-Alignment on Arabidopsis OPT3 Dataset using `salmon`

In this Notebook, you will perform a pseudo-alignment of the trimmed `fastq` files from the 6 samples trimmed and QC checked in M2P1. We will use `salmon` as our pseudo-aligner. First, you will retrieve files from the TAIR10 genome release to build a transcriptome index. This index will then be used to quantify transcript counts from our 6 samples, generating transcript-level and gene-level counts that we will use in M2E1 for DE analysis. 

**NOTE:** For further practice, you will build an environment, `Module_3_Local_env`, to utilize salmon and DESeq2 within this Notebook, and within M2E1. Please follow along with the initial steps and ensure that you are aware of the labeling of our Module 3 Bash and R kernels.

<u>**By the end of this notebook, you should have completed these tasks:** </u>
- Creation of the `Module_3` environment and activation of the `Module_3 R` kernel, and the `Module_3 Bash` kernel within Jupyter Notebook.     
- Retrieve files from TAIR's website and use them to build a `salmon` quasi (puff) index of the Arabidopsis Thaliana transcriptome.
- Run a pseudo-alignment on our 6 trimmed Col-0 and OPT3 samples from M2P1, generating `quant.sf` and `quant.genes.sf` files for each sample within the `m3_project_pseudo_alignment/results/aligned_counts/` folder.                                                                                                                                                       

```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

## Creation of the `Module_3_Local_Env` enviornment and activation of the `Module_3 R` and `Module_3 Bash` Kernels

We start this notebook within the `Bash` kernel, and we will use bash magic within the code cells to generate an environment called `Module_3_Local_Env` that contains both `irkernel`, `R`, and lots of packages for use in our pseudo-alignment pipeline. The YAML file containing the environment's packages and channels is located within the `Module_3/Resources/` folder. 

We will create this environment using bash magic and activate the kernel for `Module_3 - R` in M3E1. To use `salmon` within our `bash` kernel, we must load the `Module_3_Local_Env` using the commands `ml mamba` followed by `source activate Module_3_Local_Env`. With this, we can check if `salmon` will run.

**NOTE:** You will need a substantial amount of memory allocated within your Jupyter Notebook session, probably around 32 or 64 GB. Please be aware, and if you get a memory overflow error, restart your Jupyter Session with more memory from OnDemand.

In [ ]:
#load mamba module
ml mamba
#create env Module_3_Local_Env from yaml file
mamba env create -f ../../../environment/module_3_rnaseq_salmon.yml --yes
#activate Module_3_Local_Env
source activate Module_3_Local_Env
#R - Use R to load IR kernel with correct name and in correct path
R -e "IRkernel::installspec(name = 'Module_3_R', displayname = 'Module 3 - R')"

In [ ]:
#Load mamba
ml mamba

#activate our env
source activate Module_3_Local_Env

#This command makes the terminal known within the bash shell
export TERM=xterm

#change directory to the project folder
cd ..
cd m3_project_pseudo_alignment
#current working directory
pwd


In [ ]:
#if you run this cell, salmon should load.
salmon --help

Now, you should see the option to enable the `Module 3 - R` kernel within Jupyter Notebook. You may have to refresh the page for them to appear. 
![image.png](images/M3P2_Pseudo_Alignment_on_Arabidopsis_OPT3_Dataset_e5803a3c.png)

<u>**IMPORTANT NOTE:** </u>**FOR THE REST OF THIS NOTEBOOK USE THE `Bash` KERNEL**. 

## Retrieval of TAIR10 cDNA, Annotation, and cDNA to Transcript associations from TAIR10

If you recall from Module 2 Lab 1, a pseudo-aligner, such as `salmon`, must build an index of the transcriptome for use in mapping reads. We will use the TAIR10 cDNA for this purpose, alongside an annotation of the TAIR10 Genome in `gff` format. We will also get the Transcript Model associations with genes from TAIR10, in case we decide to use it later for the TX2gene function within `tximport`.

Visit the [TAIR10](https://www.arabidopsis.org/) website and navigate to downloads to see the available files for the TAIR10 and other *Arabidopsis Thaliana* genomes. We will pull our files using the command line to ensure that they end up in the correct folder for our salmon run (`Module_3/m3_project_pseudo_alignment/references`).

For the building of our Salmon index, we will fetch the following files: 
1. [`TAIR10_cdna_20110103_representative_gene_model_updated.fasta`](https://www.arabidopsis.org/download/file?path=Genes/TAIR10_genome_release/TAIR10_blastsets/TAIR10_cdna_20110103_representative_gene_model_updated) - The TAIR10 CDNA file
2. [`TAIR10_GFF3_genes.gff`](https://www.arabidopsis.org/download/file?path=Genes/TAIR10_genome_release/TAIR10_gff3/TAIR10_GFF3_genes.gff) - The TAIR10 gene association file
3. [`TAIR10_Model_cDNA_associations`](https://www.arabidopsis.org/api/download-files/download?filePath=Genes/TAIR10_genome_release/TAIR10_gene_transcript_associations/TAIR10_Model_cDNA_associations) - The TAIR10 Gene to Transcript association file

A READ_ME file is available for the TAIR10 Genome here: [TAIR10 README.TXT](https://www.arabidopsis.org/download/file?path=Genes/TAIR10_genome_release/README_TAIR10.txt)

In [ ]:
# Download the TAIR10 CDNA file
wget -O references/TAIR10_cdna_20110103_representative_gene_model_updated.fasta \
"https://www.arabidopsis.org/api/download-files/download?filePath=Genes/TAIR10_genome_release/TAIR10_blastsets/TAIR10_cdna_20110103_representative_gene_model_updated"

# Download the TAIR10 GFF3 gene association file
wget -O references/TAIR10_GFF3_genes.gff \
"https://www.arabidopsis.org/api/download-files/download?filePath=Genes/TAIR10_genome_release/TAIR10_gff3/TAIR10_GFF3_genes.gff"

# Download the TAIR10 Model cDNA associations
wget -O references/TAIR10_Model_cDNA_associations \
"https://www.arabidopsis.org/api/download-files/download?filePath=Genes/TAIR10_genome_release/TAIR10_gene_transcript_associations/TAIR10_Model_cDNA_associations"


Now, we should check that our folder, `m3_project_pseudo_alignment` is organized correctly: 

Are there files for each of our 6 samples within the `trimmed` folder? 

Do we see the three files we just downloaded, along with the csv and other SRA information within the `references` folder?

If so, proceed to the building of an index using `salmon`

In [ ]:
tree

## Building an Index of the TAIR10 genome with `salmon`

As we recall from M3L1, `salmon` is a fast, accurate, and bias-aware tool for quantifying transcript abundance from RNA-seq data without requiring full read alignment. It divides the transcriptome and reads into k-mers and performs a pseudo-alignment using expectation maximization to estimate when reads match transcripts, then assigns counts using little computational power in comparison to traditional aligners, like `STAR`.

**[Here is the documentation for `salmon`](https://salmon.readthedocs.io/en/latest/salmon.html)**

First, we will set the file paths for all of our files and directories. Then, we will build our index using the three files: 
1. [`TAIR10_cdna_20110103_representative_gene_model_updated.fasta`](https://www.arabidopsis.org/download/file?path=Genes/TAIR10_genome_release/TAIR10_blastsets/TAIR10_cdna_20110103_representative_gene_model_updated) - The TAIR10 CDNA file
2. [`TAIR10_GFF3_genes.gff`](https://www.arabidopsis.org/download/file?path=Genes/TAIR10_genome_release/TAIR10_gff3/TAIR10_GFF3_genes.gff) - The TAIR10 gene association file
3. [`TAIR10_Model_cDNA_associations`](https://www.arabidopsis.org/api/download-files/download?filePath=Genes/TAIR10_genome_release/TAIR10_gene_transcript_associations/TAIR10_Model_cDNA_associations) - The TAIR10 Gene to Transcript association file

The command to build an index is `salmon index -t "$TRANSCRIPTOME" -i "$INDEX_DIR" --no-version-check`, where "$TRANSCRIPTOME" is our transcriptome.


In [ ]:
# Paths
TRANSCRIPTOME="references/TAIR10_cdna_20110103_representative_gene_model_updated.fasta"
GENEMAP="references/TAIR10_Model_cDNA_associations"
TRIMMED_DIR="data/trimmed"
RESULTS_DIR="results/aligned_counts"
INDEX_DIR="references/salmon_index"

# Build Salmon index
if [ ! -d "$INDEX_DIR" ]; then
    echo "Building Salmon index..."
    salmon index -t "$TRANSCRIPTOME" -i "$INDEX_DIR" --no-version-check
else
    echo "Salmon index already exists."
fi

## Running `salmon` to align the 6 samples to the transcriptome

Now, let's run `salmon` on each of the samples within our folder `data/trimmed/` using a for loop. We are aligning our transcript counts to genes using the `gff` file [`TAIR10_GFF3_genes.gff`](https://www.arabidopsis.org/download/file?path=Genes/TAIR10_genome_release/TAIR10_gff3/TAIR10_GFF3_genes.gff).  For reference, the parameters involved in the use of `salmon` are in a quick reference table below: 
| Parameter                    | Description                                                                                  | Example/Notes                                         |
|-------------------------------|----------------------------------------------------------------------------------------------|------------------------------------------------------|
| `-i`, `--index`               | Path to the Salmon index directory                                                          | `-i salmon_index/`                                   |
| `-l`, `--libType`             | Library type (e.g., strandedness)                                                           | `-l A` (automatic detection)                        |
| `-1` and `-2`                  | Input files for paired-end reads                                                            | `-1 sample_R1.fastq.gz -2 sample_R2.fastq.gz`       |
| `-r`                          | Input file for single-end reads                                                             | `-r sample.fastq.gz`                                 |
| `--validateMappings`           | Perform selective alignment / validate mappings                                             | Often used with quasi-mapping                        |
| `--seqBias`                    | Correct for sequence-specific bias                                                          | Adds accuracy to estimates                           |
| `--gcBias`                     | Correct for GC-content bias                                                                 | Recommended for accurate transcript-level estimates |
| `--posBias`                    | Correct positional bias                                                                     | Improves uniformity of read coverage                |
| `-p`, `--threads`              | Number of threads to use                                                                    | `-p 4`                                              |
| `-o`, `--output`               | Output directory                                                                            | `-o salmon_output/`                                  |
| `--dumpEq`                     | Dump equivalence classes for downstream tools                                              | Optional                                            |
| `--geneMap`                    | Provide a transcript-to-gene mapping file                                                  | Required for gene-level summarization               |
| `--softclip`                   | Allow soft clipping during mapping                                                         | Helps with adapter contamination                     |


In [ ]:
# Quantify each sample
for fq in "$TRIMMED_DIR"/*_R1_trimmed.fastq.gz; do
    sample=$(basename "$fq" "_R1_trimmed.fastq.gz")
    outdir="$RESULTS_DIR/$sample"

    mkdir -p "$outdir"

    echo "Quantifying sample $sample..."
    salmon quant \
        -i "$INDEX_DIR" \
        -l A \
        -r "$fq" \
        --geneMap "$GENEMAP" \
        -p 4 \
        --validateMappings \
        -o "$outdir" \
        --quiet
done

## Verify that `salmon` output files in the correct location (`results/aligned_counts`)

Check that your files ended up in the correct place, your `tree` command should look something like this: 

```
├── data
│   ├── raw
│   │   ├── CRN1-1_R1.fastq.gz
│   │   ├── CRN2-1_R1.fastq.gz
│   │   ├── CRN3-1_R1.fastq.gz
│   │   ├── ORN1-1_R1.fastq.gz
│   │   ├── ORN2-1_R1.fastq.gz
│   │   └── ORN3-1_R1.fastq.gz
│   └── trimmed
│       ├── CRN1-1_R1_trimmed.fastq.gz
│       ├── CRN2-1_R1_trimmed.fastq.gz
│       ├── CRN3-1_R1_trimmed.fastq.gz
│       ├── ORN1-1_R1_trimmed.fastq.gz
│       ├── ORN2-1_R1_trimmed.fastq.gz
│       └── ORN3-1_R1_trimmed.fastq.gz
├── references
│   ├── salmon_index
│   │   ├── complete_ref_lens.bin
│   │   ├── ctable.bin
│   │   ├── ctg_offsets.bin
│   │   ├── duplicate_clusters.tsv
│   │   ├── info.json
│   │   ├── mphf.bin
│   │   ├── pos.bin
│   │   ├── pre_indexing.log
│   │   ├── rank.bin
│   │   ├── refAccumLengths.bin
│   │   ├── ref_indexing.log
│   │   ├── reflengths.bin
│   │   ├── refseq.bin
│   │   ├── seq.bin
│   │   └── versionInfo.json
│   ├── selected_samples.csv
│   ├── SraRunTable.txt
│   ├── TAIR10_cdna_20110103_representative_gene_model_updated
│   ├── TAIR10_cdna_20110103_representative_gene_model_updated.fasta
│   ├── TAIR10_GFF3_genes.gff
│   └── TAIR10_Model_cDNA_associations
├── results
│   ├── aligned_counts
│   │   ├── CRN1-1
│   │   │   ├── aux_info
│   │   │   │   ├── ambig_info.tsv
│   │   │   │   ├── expected_bias.gz
│   │   │   │   ├── fld.gz
│   │   │   │   ├── meta_info.json
│   │   │   │   ├── observed_bias_3p.gz
│   │   │   │   └── observed_bias.gz
│   │   │   ├── cmd_info.json
│   │   │   ├── lib_format_counts.json
│   │   │   ├── libParams
│   │   │   │   └── flenDist.txt
│   │   │   ├── logs
│   │   │   │   └── salmon_quant.log
│   │   │   ├── quant.genes.sf
│   │   │   └── quant.sf
│   │   ├── CRN2-1
│   │   │   ├── aux_info
│   │   │   │   ├── ambig_info.tsv
│   │   │   │   ├── expected_bias.gz
│   │   │   │   ├── fld.gz
│   │   │   │   ├── meta_info.json
│   │   │   │   ├── observed_bias_3p.gz
│   │   │   │   └── observed_bias.gz
│   │   │   ├── cmd_info.json
│   │   │   ├── lib_format_counts.json
│   │   │   ├── libParams
│   │   │   │   └── flenDist.txt
│   │   │   ├── logs
│   │   │   │   └── salmon_quant.log
│   │   │   ├── quant.genes.sf
│   │   │   └── quant.sf
│   │   ├── CRN3-1
│   │   │   ├── aux_info
│   │   │   │   ├── ambig_info.tsv
│   │   │   │   ├── expected_bias.gz
│   │   │   │   ├── fld.gz
│   │   │   │   ├── meta_info.json
│   │   │   │   ├── observed_bias_3p.gz
│   │   │   │   └── observed_bias.gz
│   │   │   ├── cmd_info.json
│   │   │   ├── lib_format_counts.json
│   │   │   ├── libParams
│   │   │   │   └── flenDist.txt
│   │   │   ├── logs
│   │   │   │   └── salmon_quant.log
│   │   │   ├── quant.genes.sf
│   │   │   └── quant.sf
│   │   ├── ORN1-1
│   │   │   ├── aux_info
│   │   │   │   ├── ambig_info.tsv
│   │   │   │   ├── expected_bias.gz
│   │   │   │   ├── fld.gz
│   │   │   │   ├── meta_info.json
│   │   │   │   ├── observed_bias_3p.gz
│   │   │   │   └── observed_bias.gz
│   │   │   ├── cmd_info.json
│   │   │   ├── lib_format_counts.json
│   │   │   ├── libParams
│   │   │   │   └── flenDist.txt
│   │   │   ├── logs
│   │   │   │   └── salmon_quant.log
│   │   │   ├── quant.genes.sf
│   │   │   └── quant.sf
│   │   ├── ORN2-1
│   │   │   ├── aux_info
│   │   │   │   ├── ambig_info.tsv
│   │   │   │   ├── expected_bias.gz
│   │   │   │   ├── fld.gz
│   │   │   │   ├── meta_info.json
│   │   │   │   ├── observed_bias_3p.gz
│   │   │   │   └── observed_bias.gz
│   │   │   ├── cmd_info.json
│   │   │   ├── lib_format_counts.json
│   │   │   ├── libParams
│   │   │   │   └── flenDist.txt
│   │   │   ├── logs
│   │   │   │   └── salmon_quant.log
│   │   │   ├── quant.genes.sf
│   │   │   └── quant.sf
│   │   └── ORN3-1
│   │       ├── aux_info
│   │       │   ├── ambig_info.tsv
│   │       │   ├── expected_bias.gz
│   │       │   ├── fld.gz
│   │       │   ├── meta_info.json
│   │       │   ├── observed_bias_3p.gz
│   │       │   └── observed_bias.gz
│   │       ├── cmd_info.json
│   │       ├── lib_format_counts.json
│   │       ├── libParams
│   │       │   └── flenDist.txt
│   │       ├── logs
│   │       │   └── salmon_quant.log
│   │       ├── quant.genes.sf
│   │       └── quant.sf
│   ├── qc
│   └── qc_trimmed
└── scripts

34 directories, 106 files
```